[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iamjamesbowden/AG952/blob/main/materials/week09/notebooks/AG952_W9_BrewDog_Facilitator.ipynb)

# AG952 | Workshop 9 -- Facilitator Debrief: BrewDog, Reading the Signals

*Dr James Bowden, University of Strathclyde Business School*

---

Run this notebook during or after the session. It reads live submissions from the class Google Sheet and produces a structured debrief with charts designed to be displayed on screen.

**What this notebook covers:**
1. Submission tracker -- which teams have submitted
2. Methodological choices -- pre-processing, LDA topics, transformer model
3. Key research question results -- which method showed the earliest signal, what interpretability tools revealed
4. Sentiment findings -- class mean scores, spread, and consensus
5. Analyst notes -- word cloud and selected quotations for class discussion

**Prerequisites:** Google Sheet populated with student data; service account credentials JSON stored in Colab Secrets as `GSHEET_CREDS`.


In [ ]:
#@title Configuration — set these before pulling data
SPREADSHEET_ID  = ""       # paste Google Sheet ID here
WORKSHEET_NAME  = "Sheet1"
APPS_SCRIPT_URL = ""      # same URL as student notebook

In [ ]:
#@title Step 0 — Install and import
!pip install gspread google-auth pandas numpy matplotlib seaborn wordcloud -q
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import json, re
from wordcloud import WordCloud
from google.colab import userdata
from google.oauth2.service_account import Credentials
import gspread
from IPython.display import display, Markdown, HTML
import warnings; warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
print("Ready.")

In [ ]:
#@title Pull data from Google Sheets
creds_json = userdata.get("GSHEET_CREDS")
creds_dict = json.loads(creds_json)
scopes = ["https://www.googleapis.com/auth/spreadsheets.readonly",
          "https://www.googleapis.com/auth/drive.readonly"]
creds = Credentials.from_service_account_info(creds_dict, scopes=scopes)
gc = gspread.authorize(creds)
ws = gc.open_by_key(SPREADSHEET_ID).worksheet(WORKSHEET_NAME)
rows = ws.get_all_records()
df = pd.DataFrame(rows)
# Clean numeric columns
for col in ["cp6_finbert_accuracy","cp6_distilbert_accuracy","cp4_sentiment_2010_2014","cp4_sentiment_2019_2025"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
print(f"Loaded {len(df)} team submissions.")
display(df.head())

In [ ]:
#@title Submission tracker -- re-run at any time to update

if df.empty:
    display(Markdown("**No submissions received yet.**"))
else:
    submitted = df.sort_values("timestamp", na_position="last") if "timestamp" in df.columns else df
    n = len(submitted)
    display(Markdown(f"### {n} team submission{'s' if n != 1 else ''} received"))
    for i, row in enumerate(submitted.itertuples(), 1):
        name = getattr(row, "team_name", "Unknown")
        ts   = str(getattr(row, "timestamp", ""))[:16]
        model = str(getattr(row, "cp6_model", "")).split(" (")[0][:20]
        print(f"  {i:>2}.  {name:<28}  {ts}  [{model}]")
    print()
    if "cp6_model" in df.columns:
        model_counts = df["cp6_model"].apply(lambda s: str(s).split(" (")[0]).value_counts()
        print("Transformer model split:")
        for m, c in model_counts.items():
            bar = "\u25a0" * c
            print(f"  {m:<28}  {bar}  ({c})")

## Part 1 -- Decision Audit

In [ ]:
#@title 1.1 — Full decision overview
# Note: cp4_dictionary stores the reflection answer from CP4 --
# "Which method showed the earliest and clearest signal of distress?"
cols = ["team_name","cp1_period","cp2_normalisation","cp2_stopwords",
        "cp3_n_topics","cp4_dictionary","cp6_model","cp7_interpretability_choice"]
avail = [c for c in cols if c in df.columns]
display(
    df[avail]
    .rename(columns={"cp4_dictionary": "earliest_signal_method"})
    .to_html(index=False)
)
for col, label in [
    ("cp2_normalisation",          "Normalisation method"),
    ("cp2_stopwords",              "Stop-word list"),
    ("cp4_dictionary",             "Earliest-signal method (CP4 reflection)"),
    ("cp6_model",                  "Transformer model choice"),
    ("cp7_interpretability_choice","IG assessment (CP5 reflection)"),
]:
    if col in df.columns:
        print(f"\n{label}:")
        print(df[col].value_counts().to_string())

In [ ]:
#@title 1.2 — Pre-processing decision heatmap
if "cp2_normalisation" in df.columns and "cp2_stopwords" in df.columns:
    ct = pd.crosstab(df["cp2_normalisation"], df["cp2_stopwords"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.heatmap(ct, annot=True, fmt="d", cmap="YlOrRd", ax=axes[0], cbar=False)
    axes[0].set_title("Pre-processing choices: normalisation \u00d7 stop-words\n(cell = number of teams)")
    axes[0].set_xlabel("Stop-word list"); axes[0].set_ylabel("Normalisation")

    # Stacked bar: transformer model by normalisation
    if "cp6_model" in df.columns:
        model_short = df["cp6_model"].str.split(" (").str[0]
        ct2 = pd.crosstab(df["cp2_normalisation"], model_short)
        ct2.plot(kind="bar", stacked=True, ax=axes[1], colormap="tab10", edgecolor="white")
        axes[1].set_title("Transformer model choice by normalisation method\n")
        axes[1].set_xlabel("Normalisation"); axes[1].set_ylabel("Teams")
        axes[1].tick_params(axis="x", rotation=25)
        axes[1].legend(title="Model", fontsize=8)
    plt.tight_layout(); plt.show()

In [ ]:
#@title 1.3 — LDA topic count choices (k) and model selection
if "cp3_n_topics" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df["cp3_n_topics_num"] = pd.to_numeric(df["cp3_n_topics"], errors="coerce")
    axes[0].hist(df["cp3_n_topics_num"].dropna(), bins=range(2, 14), color="#4f46e5",
                 edgecolor="white", alpha=0.85)
    axes[0].set_title("Distribution of k (number of LDA topics) across teams")
    axes[0].set_xlabel("k"); axes[0].set_ylabel("Teams")

    if "cp6_model" in df.columns:
        model_counts = df["cp6_model"].apply(lambda s: str(s).split(" (")[0]).value_counts()
        axes[1].pie(
            model_counts.values,
            labels=model_counts.index,
            autopct="%1.0f%%",
            colors=["#6366f1", "#10b981", "#f59e0b"],
        )
        axes[1].set_title("Transformer model selection")
    plt.tight_layout(); plt.show()

    n_valid = df["cp3_n_topics_num"].notna().sum()
    if n_valid > 0:
        print(f"\nMedian k:  {df['cp3_n_topics_num'].median():.0f}")
        print(f"Range:     {df['cp3_n_topics_num'].min():.0f} \u2013 {df['cp3_n_topics_num'].max():.0f}")
        print(f"Responses: {n_valid}")


In [ ]:
#@title 1.3b — RQ2 result: which method showed the earliest signal?
# This is the key RQ2 finding -- display prominently for class discussion.
if "cp4_dictionary" in df.columns:
    # Shorten labels for display
    short = {
        "LM dictionary showed the earliest signal":         "LM dictionary",
        "VADER showed the earliest signal":                 "VADER",
        "FinBERT showed the earliest signal":               "FinBERT",
        "DistilBERT showed the earliest signal":            "DistilBERT",
        "All methods agreed -- signal was consistent":      "All methods agreed",
        "Methods diverged -- results were inconsistent":    "Methods diverged",
        "None of the methods showed a clear early signal":  "No clear signal",
        "-- select after seeing the results --":            "(not answered)",
    }
    answered = df["cp4_dictionary"][df["cp4_dictionary"] != "-- select after seeing the results --"]
    if len(answered) == 0:
        print("No responses recorded yet.")
    else:
        counts = answered.map(lambda s: short.get(s, str(s)[:35])).value_counts()
        palette = {
            "LM dictionary":       "#f59e0b",
            "VADER":               "#4f46e5",
            "FinBERT":             "#6366f1",
            "DistilBERT":          "#10b981",
            "All methods agreed":  "#22c55e",
            "Methods diverged":    "#ef4444",
            "No clear signal":     "#94a3b8",
        }
        bar_colours = [palette.get(k, "#cbd5e1") for k in counts.index]
        fig, ax = plt.subplots(figsize=(11, max(3, len(counts) * 0.65)))
        bars = ax.barh(counts.index, counts.values, color=bar_colours, edgecolor="white", height=0.6)
        ax.set_title("RQ2 class verdict: which method revealed distress earliest?\n(CP4 reflection, all teams)",
                     fontsize=12)
        ax.set_xlabel("Number of teams")
        ax.set_xlim(0, counts.max() + 1.5)
        for bar, v in zip(bars, counts.values):
            ax.text(v + 0.15, bar.get_y() + bar.get_height() / 2, str(v),
                    va="center", fontsize=12, fontweight="bold")
        ax.tick_params(axis="y", labelsize=10)
        plt.tight_layout(); plt.show()
        print(f"\n{len(answered)} team{'s' if len(answered) != 1 else ''} answered.")

In [ ]:
#@title 1.4 — Student assessment of integrated gradients (CP5)
if "cp7_interpretability_choice" in df.columns:
    def simplify(s):
        s = str(s)
        if "clear and informative" in s:   return "IG: clear and informative"
        if "hard to generalise"    in s:   return "IG: useful but limited"
        if "Attention weights"     in s:   return "Prefer attention weights"
        if "too opaque"            in s:   return "Model too opaque"
        if "Need both"             in s:   return "Need both methods"
        return "Not answered"
    df["interp_simplified"] = df["cp7_interpretability_choice"].apply(simplify)
    counts = df["interp_simplified"].value_counts()
    colours = {
        "IG: clear and informative": "#22c55e",
        "IG: useful but limited":    "#86efac",
        "Prefer attention weights":  "#f59e0b",
        "Model too opaque":          "#ef4444",
        "Need both methods":         "#4f46e5",
        "Not answered":              "#d1d5db",
    }
    bar_colours = [colours.get(k, "#9ca3af") for k in counts.index]
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(counts.index, counts.values, color=bar_colours, edgecolor="white")
    ax.set_title("Are integrated gradients a reliable tool for financial text analysis?\n(CP5 student assessment)")
    ax.set_xlabel("Teams"); ax.set_xlim(0, counts.max() + 1.5)
    for bar, v in zip(bars, counts.values):
        ax.text(v + 0.15, bar.get_y() + bar.get_height() / 2, str(v), va="center", fontsize=11, fontweight="bold")
    plt.tight_layout(); plt.show()
    print("\nDiscussion: did teams that chose FinBERT (domain-specific) rate IG more favourably")
    print("than teams using DistilBERT? Jain and Wallace (2019) showed attention is unreliable --")
    print("did students observe the same concern with gradients on the tricky sentences (3 and 4)?")

## Part 2 -- Results Analysis

In [ ]:
#@title 2.1 — Reported VADER sentiment scores: growth era vs crisis
# cp4_dictionary stores the reflection: "which method showed the earliest and clearest signal?"
cols_sent = ["cp4_sentiment_2010_2014","cp4_sentiment_2019_2025"]
avail_sent = [c for c in cols_sent if c in df.columns and df[c].notna().any()]
if avail_sent:
    sent_df = df[
        ["team_name"] + avail_sent + (["cp4_dictionary"] if "cp4_dictionary" in df.columns else [])
    ].dropna(subset=avail_sent)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter: early vs crisis -- colour by earliest-signal method
    if "cp4_dictionary" in df.columns and len(sent_df) > 0:
        # Shorten option labels for the legend
        sent_df = sent_df.copy()
        sent_df["_signal"] = sent_df["cp4_dictionary"].str.replace(" showed the earliest signal", "", regex=False)
        choices = sent_df["_signal"].unique()
        palette = ["#4f46e5","#f59e0b","#10b981","#ef4444","#8b5cf6","#06b6d4","#f43f5e"]
        colours = dict(zip(choices, palette))
        for choice, grp in sent_df.groupby("_signal"):
            axes[0].scatter(
                grp["cp4_sentiment_2010_2014"], grp["cp4_sentiment_2019_2025"],
                label=str(choice)[:30], color=colours.get(choice, "grey"), s=90, alpha=0.85,
            )
        axes[0].legend(title="Earliest signal", fontsize=7, loc="best")
    else:
        axes[0].scatter(
            sent_df["cp4_sentiment_2010_2014"], sent_df["cp4_sentiment_2019_2025"],
            color="#4f46e5", s=90, alpha=0.85,
        )
    axes[0].axhline(0, color="grey", lw=0.8, linestyle="--")
    axes[0].axvline(0, color="grey", lw=0.8, linestyle="--")
    axes[0].set_xlabel("Mean VADER score 2010\u20132014 (growth)")
    axes[0].set_ylabel("Mean VADER score 2019\u20132025 (crisis)")
    axes[0].set_title("VADER sentiment: growth era vs crisis\n(coloured by earliest-signal method)")

    # Bar: class mean by period
    means = sent_df[avail_sent].mean()
    axes[1].bar(
        ["2010\u20132014\n(growth)", "2019\u20132025\n(crisis)"],
        means.values,
        color=["#22c55e", "#ef4444"], edgecolor="white", width=0.5,
    )
    axes[1].axhline(0, color="grey", lw=0.8, linestyle="--")
    axes[1].set_title("Class mean VADER sentiment by period")
    axes[1].set_ylabel("Mean VADER compound score")
    for i, v in enumerate(means.values):
        axes[1].text(i, v + (0.005 if v >= 0 else -0.012), f"{v:+.3f}", ha="center", fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()
    print(f"\nClass mean 2010\u20132014:  {sent_df['cp4_sentiment_2010_2014'].mean():+.4f}  (n={sent_df['cp4_sentiment_2010_2014'].notna().sum()})")
    print(f"Class mean 2019\u20132025:  {sent_df['cp4_sentiment_2019_2025'].mean():+.4f}  (n={sent_df['cp4_sentiment_2019_2025'].notna().sum()})")
    diff = sent_df["cp4_sentiment_2019_2025"].mean() - sent_df["cp4_sentiment_2010_2014"].mean()
    print(f"Change:               {diff:+.4f}")

In [ ]:
#@title 2.2 — VADER-transformer agreement rates across teams
# cp6_finbert_accuracy = fraction of articles where FinBERT and VADER agreed on sentiment direction.
# cp6_distilbert_accuracy = same for DistilBERT-SST2.
# Agreement < 70% suggests the two methods are measuring different things.
acc_cols = {c: c.replace("cp6_","").replace("_accuracy","").replace("_"," ").title()
            for c in ["cp6_finbert_accuracy","cp6_distilbert_accuracy"] if c in df.columns}
if acc_cols:
    model_colours = {"Finbert": "#6366f1", "Distilbert": "#10b981"}
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for (col, label), ax_i in zip(acc_cols.items(), range(2)):
        vals = (df[col].dropna() * 100).round(1)
        if len(vals) == 0:
            continue
        colour = model_colours.get(label.split()[0], "#94a3b8")
        axes[ax_i].hist(vals, bins=max(5, len(vals) // 2), color=colour, edgecolor="white", alpha=0.88)
        axes[ax_i].axvline(vals.mean(), color="red", lw=1.8, linestyle="--",
                           label=f"Mean {vals.mean():.1f}%")
        axes[ax_i].axvline(70, color="#94a3b8", lw=1.2, linestyle=":",
                           label="70% threshold")
        axes[ax_i].set_title(f"{label} vs VADER\n(% of articles -- same sentiment direction)")
        axes[ax_i].set_xlabel("Agreement rate (%)"); axes[ax_i].set_ylabel("Teams")
        axes[ax_i].legend(fontsize=9)
    plt.tight_layout(); plt.show()

    print("\nMean VADER-agreement rate by model:")
    for col, label in acc_cols.items():
        vals = df[col].dropna()
        if len(vals):
            print(f"  {label:<20}  {vals.mean():.1%}  (range {vals.min():.1%}\u2013{vals.max():.1%},  n={len(vals)} teams)")

    print()
    print("Discussion: FinBERT is trained on financial news; DistilBERT is trained on general text.")
    print("If FinBERT agrees with VADER more often, it may simply share VADER's lexicon of positive")
    print("financial terms. If it agrees less, it may be picking up domain-specific nuance that VADER")
    print("misses -- whether that is better or worse depends on the application.")
    if all(c in df.columns for c in acc_cols):
        fin_mean = df["cp6_finbert_accuracy"].mean()
        dis_mean = df["cp6_distilbert_accuracy"].mean()
        if not (pd.isna(fin_mean) or pd.isna(dis_mean)):
            diff = fin_mean - dis_mean
            direction = "higher" if diff > 0 else "lower"
            print(f"\n  FinBERT agreement is {abs(diff):.1%} {direction} than DistilBERT on average.")

In [ ]:
#@title 2.4 — Within-class methodological divergence analysis
display(Markdown("""
### 2.4 -- Divergence analysis: does methodology drive outcome?

Key questions for class discussion:
- Did teams that identified **FinBERT** as the earliest-signal method report different VADER crisis-period scores than teams that picked **VADER** or **LM dictionary**?
- Did teams choosing **FinBERT** show higher VADER-agreement rates than **DistilBERT** teams?
- Did teams that chose **more topics (higher k)** tend to describe more specific BrewDog themes in their analyst notes, or did smaller k produce more coherent narratives?
- Is there a relationship between **pre-processing strictness** (lemmatisation plus finance-adjusted stop words) and the **VADER sentiment scores** reported?
- Did teams with **higher VADER-transformer agreement** rate IG as more reliable (CP5)? This would suggest that agreement between methods builds confidence in both.
"""))

# Table: crisis-period VADER by earliest-signal method
if "cp4_dictionary" in df.columns and "cp4_sentiment_2019_2025" in df.columns:
    grp = (
        df.groupby("cp4_dictionary")["cp4_sentiment_2019_2025"]
        .agg(["mean","std","count"])
        .rename(columns={"mean": "Mean VADER (2019\u20132025)", "std": "Std dev", "count": "Teams"})
    )
    grp["Mean VADER (2019\u20132025)"] = grp["Mean VADER (2019\u20132025)"].round(4)
    grp["Std dev"] = grp["Std dev"].round(4)
    grp.index.name = "Earliest-signal method"
    print("\nCrisis-period VADER sentiment grouped by which method teams thought\nshowed the earliest signal of distress:")
    display(grp)

# Table: VADER-agreement by transformer model
if "cp6_model" in df.columns and any(c in df.columns for c in ["cp6_finbert_accuracy","cp6_distilbert_accuracy"]):
    print("\nVADER-agreement rate by transformer model selected:")
    for col, label in [("cp6_finbert_accuracy","FinBERT"), ("cp6_distilbert_accuracy","DistilBERT")]:
        if col in df.columns:
            grp2 = df.groupby(df["cp6_model"].str.split(" (").str[0])[col].agg(["mean","count"]).rename(columns={"mean": f"{label} agreement", "count": "Teams"})
            grp2[f"{label} agreement"] = grp2[f"{label} agreement"].apply(lambda x: f"{x:.1%}" if pd.notna(x) else "")
            display(grp2)

## Part 3 -- Analyst Notes and Synthesis

In [ ]:
#@title 3.1 — Analyst note word cloud and quotations
if "cp8_analyst_note" in df.columns:
    notes = " ".join(df["cp8_analyst_note"].dropna().tolist())
    if notes.strip():
        stop = {
            "the","a","an","and","of","to","in","is","are","was","were","for","that",
            "this","with","have","has","had","be","been","but","we","our","their",
            "would","could","should","may","might","also","its","it","at","by","not",
            "more","than","as","on","from","or","i","which","between","however",
        }
        wc = WordCloud(
            width=1100, height=400, background_color="white", colormap="RdYlGn",
            max_words=90, stopwords=stop, min_font_size=10,
        ).generate(notes)
        fig, ax = plt.subplots(figsize=(13, 4.5))
        ax.imshow(wc, interpolation="bilinear"); ax.axis("off")
        ax.set_title("Class analyst notes -- word cloud", fontsize=13, pad=10)
        plt.tight_layout(); plt.show()

    # Print each note for projection
    print("\nIndividual analyst notes:\n" + "="*70)
    for _, row in df[["team_name","cp8_analyst_note"]].dropna().iterrows():
        if str(row["cp8_analyst_note"]).strip():
            print(f"\n[{row['team_name']}]")
            print(row["cp8_analyst_note"].strip())
            print("-"*50)

In [ ]:
#@title 3.2 — Session synthesis: three research questions answered by the class
display(Markdown("""
### Session synthesis

Use the results above to anchor the debrief around the three research questions.

---

**RQ1. Coverage and topic composition**

- CP1 shows the volume shift: coverage accelerates sharply from 2019 onwards as controversy begins.
- CP3 LDA: look at whether students found coherent topics within each era, and whether the topics recovered the known narrative (crowdfunding growth, workplace scandal, administration risk).
- Ask: did students choosing a higher k find more granular but noisier topics, or more useful ones?

---

**RQ2. Do methods agree on the timing and direction of sentiment?**

- Cell 1.3b (above) shows the class verdict on which method detected distress earliest.
- Cell 2.1 shows whether VADER scores in the crisis period are negative on average and whether the drop from 2010-2014 is consistent across teams.
- Cell 2.2 shows VADER-transformer agreement. Low agreement does not mean one method is wrong -- it means they are capturing different things.
- Key point: FinBERT was trained on Bloomberg and Reuters news; its sensitivity to financial distress vocabulary may explain any divergence from VADER on culture-scandal articles.

---

**RQ3. Are transformer explanations reliable enough for regulated financial analysis?**

- Cell 1.4 shows the class assessment. If most teams chose "useful but hard to generalise", ask why five sentences is not sufficient.
- Refer to sentences 3 and 4: if FinBERT correctly labelled "risk had been materially reduced" as positive, and the IG attribution showed "reduced" and "refinancing" as key drivers, that is a strong result.
- If teams observed high attribution on stopwords or punctuation, that supports the "hard to generalise" position.
- Close with the regulatory implication: the EU AI Act requires explainability for high-risk financial AI. What standard of evidence would be needed to satisfy that requirement?
"""))